IT12

In [ ]:
import os

# Create folder if it does not exist
os.makedirs("data", exist_ok=True)

# Production records, including three deliberately faulty records
records = [
    "P101,25.02,DAY",
    "P102,24.93,DAY",
    "P103,25.15,DAY",
    "P104,24.98,DAY",
    "P105,25.12,NIGHT",
    "P106,25.01,NIGHT",
    "P107,24.85,NIGHT",
    "P108,25.04,DAY",
    "P109,24.96,DAY",
    "P110,25.08,NIGHT",
    "P111,abc,DAY",       # Fault 1: non-numeric diameter
    "P112,25.02",         # Fault 2: missing shift
    "P113,,NIGHT",        # Fault 3: missing diameter
    "P114,25.00,DAY",
    "P115,24.99,NIGHT"
]

# Write records to file
with open("data/shift_log.txt", "w") as file:
    for record in records:
        file.write(record + "\n")

print("Shift log created successfully.")

# Read the file back
with open("data/shift_log.txt", "r") as file:
    lines = file.readlines()

print("\nFile contents:")
for line in lines:
    print(line.strip())

print("\nTotal lines:", len(lines))

IT10

In [ ]:
def parse_line(line):
    try:
        if not line:
            raise ValueError("empty line")

        parts = line.split(",")

        if len(parts) != 2:
            raise ValueError("wrong number of fields")

        code = parts[0].strip()
        dia = float(parts[1])

        if not code:
            raise ValueError("missing part code")

        if dia <= 0:
            raise ValueError("diameter must be positive")

        return code, dia

    except ValueError as e:
        return None, str(e)


valid = []
rejects = []

for line in hostile:
    code, result = parse_line(line)

    if code is not None:
        valid.append((code, result))
    else:
        rejects.append((line, result))

if valid:
    average = sum(dia for code, dia in valid) / len(valid)
else:
    average = 0

print("REPORT")
print("Valid parts:", len(valid))
print("Average:", round(average, 2))

print("\nREJECTS")
for line, reason in rejects:
    print(line, "->", reason)

IT8

In [ ]:
class SpecViolation(Exception):
    pass


class Machine:
    def __init__(self, name):
        self.name = name
        self.__parts = 0

    def run(self, job):
        code, dia = job

        if dia < 24.90 or dia > 25.10:
            raise SpecViolation("diameter out of specification")

        self.__parts += 1
        print(code, "processed")

    def status(self):
        return self.__parts


machine = Machine("Machine 1")

jobs = [
    ("P01", 25.00),
    ("P02", 24.95),
    ("P03", 25.20)
]

for job in jobs:
    try:
        machine.run(job)
    except SpecViolation as e:
        print(job[0], "rejected:", e)

print("Status:", machine.status())

IT6

In [ ]:
import re
import math


def parse_log(line):
    """Validates a record and returns its fields."""

    if not isinstance(line, str):
        raise TypeError("Record must be text")

    pattern = (
        r"^\s*(P\d{3})\s*,\s*"
        r"(\d+(?:\.\d+)?)\s*,\s*"
        r"(DAY|NIGHT)\s*$"
    )

    match = re.fullmatch(pattern, line, re.IGNORECASE)

    if not match:
        raise ValueError("Invalid record format")

    code, dia_text, shift = match.groups()
    dia = float(dia_text)

    if not math.isfinite(dia) or dia <= 0:
        raise ValueError("Invalid diameter")

    return code, dia, shift.upper()


def inspect(dia):
    """Classifies a valid diameter reading."""

    if 24.95 <= dia <= 25.05:
        return "ACCEPT"

    elif 24.90 <= dia <= 25.10:
        return "REWORK"

    return "SCRAP"


def process_file(path):
    """Processes one file and returns its inspection results."""

    counts = {
        "ACCEPT": 0,
        "REWORK": 0,
        "SCRAP": 0
    }

    diameters = []
    rejects = []

    with open(path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    for line in lines:

        raw = line.rstrip("\r\n")

        try:
            code, dia, shift = parse_log(raw)

            category = inspect(dia)
            counts[category] += 1
            diameters.append(dia)

        except (ValueError, TypeError) as error:
            rejects.append([raw, str(error)])

    average = (
        sum(diameters) / len(diameters)
        if diameters else None
    )

    return {
        "filename": path.name,
        "records": len(lines),
        "counts": counts,
        "average": average,
        "rejects": rejects
    }


def write_report(result, path):
    """Writes an aligned individual production report."""

    counts = result["counts"]

    lines = [
        "=" * 42,
        "          PRODUCTION FILE REPORT",
        "=" * 42,
        f"File: {result['filename']}",
        "-" * 28,
        f"{'CATEGORY':<18}{'COUNT':>10}",
        "-" * 28,
        f"{'ACCEPT':<18}{counts['ACCEPT']:>10}",
        f"{'REWORK':<18}{counts['REWORK']:>10}",
        f"{'SCRAP':<18}{counts['SCRAP']:>10}",
        "-" * 28,
        f"{'RECORDS':<18}{result['records']:>10}",
        f"{'REJECTS':<18}{len(result['rejects']):>10}",
        f"{'AVERAGE':<18}{result['average'] if result['average'] is not None else 'N/A':>10}",
        "=" * 42
    ]

    with open(path, "w", encoding="utf-8") as file:
        file.write("\n".join(lines) + "\n")


def write_plant_summary(results, path):
    """Combines results from all processed files."""

    plant = {
        "ACCEPT": 0,
        "REWORK": 0,
        "SCRAP": 0
    }

    total_records = 0
    total_rejects = 0
    total_valid = 0
    total_diameter = 0.0

    for result in results:

        for category in plant:
            plant[category] += result["counts"][category]

        total_records += result["records"]
        total_rejects += len(result["rejects"])

        valid_count = sum(result["counts"].values())

        total_valid += valid_count

        if result["average"] is not None:
            total_diameter += result["average"] * valid_count

    average = (
        total_diameter / total_valid
        if total_valid else None
    )

    lines = [
        "=" * 42,
        "           COMBINED PLANT REPORT",
        "=" * 42,
        f"Files processed: {len(results)}",
        f"Total records: {total_records}",
        f"Valid records: {total_valid}",
        f"Rejected records: {total_rejects}",
        "-" * 28,
        f"ACCEPT: {plant['ACCEPT']}",
        f"REWORK: {plant['REWORK']}",
        f"SCRAP: {plant['SCRAP']}",
        f"Average diameter: {average if average is not None else 'N/A'}",
        "=" * 42
    ]

    with open(path, "w", encoding="utf-8") as file:
        file.write("\n".join(lines) + "\n")

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import shoptools


# Input and output folders
data_folder = Path("data")
report_folder = data_folder / "reports"

report_folder.mkdir(parents=True, exist_ok=True)


# Find all input .txt files
# Exclude the IT10 report and the reports subfolder
input_files = sorted(
    path for path in data_folder.glob("*.txt")
    if path.name != "report.txt"
)


# Process all input files
results = []

for path in input_files:

    result = shoptools.process_file(path)
    results.append(result)

    output_path = report_folder / f"{path.stem}_report.txt"

    shoptools.write_report(result, output_path)


# Generate combined plant report
shoptools.write_plant_summary(
    results,
    report_folder / "plant_summary.txt"
)


# Append run history
history_file = data_folder / "run_history.csv"

with open(history_file, "a", newline="", encoding="utf-8") as file:

    writer = csv.writer(file)

    if history_file.stat().st_size == 0:
        writer.writerow([
            "Timestamp",
            "Filename",
            "Records Processed",
            "Rejects"
        ])

    timestamp = datetime.now().astimezone().isoformat(
        timespec="seconds"
    )

    for result in results:

        writer.writerow([
            timestamp,
            result["filename"],
            result["records"],
            len(result["rejects"])
        ])


print("BATCH PROCESSING COMPLETED")
print("Files processed:", len(results))
print("Individual reports:", report_folder)
print("Plant summary:", report_folder / "plant_summary.txt")
print("Run history:", history_file)